# HW 3

FAA provides high-quality data for researchers to analyze flight delays in all airports nation wide

https://aspm.faa.gov/tfms/sys/Airport.asp Links to an external site.

To fetch delay information, you need to

1) Select the date you want to query:



<img src="https://www.dropbox.com/scl/fi/l4h75887ffk0y88ocn2ck/DateSelect.png?rlkey=2x1rqqqvpnzel6jvw6142lf8c&st=5gf1vkbo&raw=1" width="400">



2) Type in the interested airport(s) ICAO ID(s)



<img src="https://www.dropbox.com/scl/fi/kojvy7e72puy10yun8ko8/airport_MCO.png?rlkey=bo211ct7zoai7v4gring10l6c&st=8r1n6x08&raw=1" width="400">

3) Select which field you want to aggregate (I select as many fields as possible so that I can fetch data fine granularity ):

<img src="https://www.dropbox.com/scl/fi/0bsfnh7kpfwq2wtpbjk9w/groupings.png?rlkey=tdnkfj6p0gae19g3e3gqm8a6r&st=c6l9z1u2&raw=1" width="400">


4) Click the Run icon on the top right corner. You should see a web page like below.

<img src="https://www.dropbox.com/scl/fi/hnsemzcmxlkzlv4erc2p0/result1.png?rlkey=542jmtf9oott9fdgyrsvjl2i3&st=6sj5g81p&raw=1" width="800">


# Questions (Screenshots and text illustrations are needed to prove your conclusion, please assemble your answers in one colab notebook)

*a) In the newly opened page in step 4, what will happen if you refresh the page? What HTTP request (GET/POST) is sent to the FAA server? Can you prove that? Hint: You may need to open the web browser's Developer Tools and then click the Refresh button.*



**ANSWER: Refreshing the page sends the POST request as shown in the header section in the Network Tab (in browser’s Developer Tool).**



<img src="https://www.dropbox.com/scl/fi/4oftlvrveh65xm3kpgels/result_POST.png?rlkey=qzca7jxw6ygp43x7jphabe9h4&st=m4tlyige&raw=1" width="600">


*(b) Please analyze the request send by your browser in step 4, can you generate a similar request using the request library and obtain a similar response?*

Yes, we can generate a similar request using Python's requests library. For that we need to examine the HTTP request sent by the browser. Examining the Browser’s Developer Tools > "Network" tab, we can generate the request that was sent when we clicked "Run" button in the webpage. Next, we need to examine different parts of Headers ( General, Response Headers, Request Headers) for information like
- Request URL  
- Request Method  ( i.e. POST)
- Request Headers ( e.g.  User-Agent, Content-Type,  cookies, etc)
- Payload for the request

<img src="https://www.dropbox.com/scl/fi/p361pzhmfathqvrcbfcq9/Network_portions.png?rlkey=4b2ww5qc7gy79flanwntfvnex&st=d43f93l5&raw=1" width="900">


*(c) Write an automated program to fetch flight delay report for one week around an airport you select.*

In [43]:
from google.colab import drive
drive.mount('/content/drive/')

import os

os.chdir('/content/drive/My Drive/GU/3_webAutomation/')

Mounted at /content/drive/


## Generalized function

In [44]:
import pandas as pd
import requests
import time
import io


In [45]:
def generate_faa_delay_reports(
    airport_code,
    header_string,
    cookie_string,
    payload_string,
    output_csv="FAA_Delay_Report.csv",
    wait_seconds=3
):
    """
    Fetch and clean FAA delay reports using provided strings.

    Parameters:
        airport_code (str): ICAO airport code (e.g., 'JFK', 'MCO')
        header_string (str): Raw HTTP header string from browser
        cookie_string (str): Raw cookie string from browser
        payload_string (str): Payload with placeholders {DATES} and {AIRPORT}
        output_csv (str): Output CSV file name
        wait_seconds (int): Time in seconds to wait before request
    """


    # Convert header string to dictionary
    header_lines = header_string.strip().splitlines()
    header_dict = {}
    for line in header_lines:
        if ": " in line:
            key, val = line.split(": ", 1)
            header_dict[key] = val

    # Convert cookie string to dictionary
    cookie_parts = cookie_string.strip().split("; ")
    cookie_dict = {}
    for item in cookie_parts:
        if "=" in item:
            key, val = item.split("=", 1)
            cookie_dict[key] = val

    # Prepare payload
    payload_lines = payload_string.strip().splitlines()
    payload_dict = {}
    for line in payload_lines:
        if ": " in line:
            key, val = line.split(": ", 1)
            payload_dict[key] = val
        else:
            payload_dict[line] = ""

    # Wait if needed
    time.sleep(wait_seconds)

    try:
        response = requests.post(
            "https://aspm.faa.gov/tfms/sys/tfms-server-x.asp",
            headers=header_dict,
            cookies=cookie_dict,
            data=payload_dict
        )

        if response.status_code == 200:
            tables = pd.read_html(io.StringIO(response.text), header=1)

            if tables:
                df = tables[0]

                # Remove rows with non-numeric "#" values and make safe copy
                if df.columns[0] == "#":
                    df = df[df["#"].apply(lambda x: str(x).isdigit())].copy()
                    df["AirportCode"] = airport_code

                df.to_csv(output_csv, index=False)
                print(f"\n Cleaned data saved to '{output_csv}'")
                print(f"\n Total records saved: '{len(df)}'")
                display(df.head())
            else:
                print(f" No table found in the response for {airport_code}")
        else:
            print(f" Request failed: HTTP {response.status_code}")
    except Exception as e:
        print(f" Exception occurred: {e}")


In [46]:
header_str =  """ POST /tfms/sys/tfms-server-x.asp HTTP/1.1
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7
Accept-Encoding: gzip, deflate, br, zstd
Accept-Language: en-US,en;q=0.9
Cache-Control: max-age=0
Connection: keep-alive
Content-Length: 1611
Content-Type: application/x-www-form-urlencoded
Host: aspm.faa.gov
Origin: https://aspm.faa.gov
Referer: https://aspm.faa.gov/tfms/sys/Airport.asp
Sec-Fetch-Dest: document
Sec-Fetch-Mode: navigate
Sec-Fetch-Site: same-origin
Sec-Fetch-User: ?1
Upgrade-Insecure-Requests: 1
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36
sec-ch-ua: "Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"
sec-ch-ua-mobile: ?0
sec-ch-ua-platform: "macOS"
"""

cookie_str = """ASPSESSIONIDQSBCSBDD=FFEFIKBANAJOPIGFDNHEAOOI; ak_bmsc=B055A54F8E8CA9D2A9F06B6A6EA67443~000000000000000000000000000000~YAAQHwVaaGxqUY2XAQAAT7OYmByMvOl7rblMY1RsKcjgHFVtLnWFCTjB5XS7SVNfghaPbPf+54xDu09xf10XLQFzsGBljd84es7J1BD8Q5ruuqa1P7mhBpASzmbIE9SgO/4YQMZk4LMFSYXhGu6Ccx/hBkOfBCaEm/BWMY7NYdhQxgNT/kD3BgfmAKsIsvrmuvKPXySBFNq+RZBIr9LM8vNwzVeMbAmKNJTXuk6muG0FiAKXeVSiSnTSsw4YHMuGvneuKCgnC+Dd4ITWZSfuCq7J6c5ZsL/w9FLTMsv+FMTV1UFNP5rgCJIO1adZ21qb/u6S8f865oZ34AyjEbls/vPRAOBb2WfyeKrfwDBQdA3lgMQM/4kQpnnuluU=; ASPMFAAGOV=LOGIN%5FDATE=06%2F22%2F2025+13%3A03%3A53&ADVISORY=NO&DP1%5FACCESS=N&ATC%5FLEVEL=N&RI%5FINPUT=N&OPSNET%5FLEVEL=4&ASPM%5FPERSPECTIVE=3&ASPM%5FAIRCOMMFAC=FAA&ASPM%5FAIRCOMM=N&NAS%5FRECAP%5FACCESS=N&ASPM%5FWCOMM=N&VP%5FACCESS=N&NY%5FWEEKLY=N&DP2%5FACCESS=N&RW%5FCONF=2&GROUP%5FID=OUT&LOG=N&APM%5FCOMMENTS=N&COUNTOPS=N&SIGNER=0&ASQP%5FLEVEL=2&SYSADMIN=N&PHONE=1234567890&LNAME=GUEST&PERTI=NO&ATCSCC=N&ASPM%5FCANCELLATIONS=N&ASPM%5FLEVEL=4&VALIDATED=Y&COMPANY=FAA+GUEST&INDEV%5FACCESS=N&SNAPSHOT%5FACCESS=N&INTERNAL=N&ATC%5FOK=True&PWORD=FAAGUEST&APM%5FDASHBOARD=N&ASPM%5FSCENARIO=N&ASPM%5FCAUSALITY=1&ASPM%5FRCALC=N&RESERVED=OTH&SYSUSER=FAAGUEST&AERO=N&OPR%5FACCESS=N&COD%5FACCESS=N&ETMS%5FLEVEL=2&FSDS%5FLEVEL=9&ASPM%5FENROUTE=N&EMAIL=FAA%40GUEST%2EGOV&FNAME=FAA&WUID=2059; ASPSESSIONIDQQDDRDCD=HEGJMPPDPFLPIKJFMFKEBLBE; ASPSESSIONIDQQBDSADC=COGNFNBAGGDJBHFLJAJHFHOG; bm_sv=49F07927D1A1B7D1AD7F21EAF73804A7~YAAQnJbTZzFWuniXAQAAmfSwmBywwmo6lzSS+pxnPB+PLuFBA1TMEID17bUK25m5jpWBRXyJEOJzL3joLBJQV+t6NoLGL8Gk1c3ZNUtAqmtDVGfcSZFEiSjN6FOgDWhuAXGttvUOLz/4xd/iUZtjnz+HfJrlwbxW4dHFmmtr+dhomvGg4AVdvuF9JJqSO9fYiXya8K+xiE1t9BeDHAkeG8JqZKK8Mbv7wf2O5AnvdE8YrHG5bD8ubD4hH4uKAw==~1"""


payload_template_RAW = """dstyle: d
dfld: yyyymmdd
dlist: 20250424,20250425,20250426,20250427,20250428,20250429,20250430
fromdate:
todate:
llist: ' MCO'
elist:
keylist: yyyymmdd,LOCID,FLT_TYPE,USER_CLASS,WEIGHT_CLASS,PHYS_CLASS,ETMS_EQPT,B_JET,R_JET,BUSAVIATION,AAC,ADG,TDG
line: SELECT YYYYMMDD,LOCID,FLT_TYPE,USER_CLASS,WEIGHT_CLASS,PHYS_CLASS,ETMS_EQPT,B_JET,R_JET,BUSAVIATION,AAC,ADG,TDG ,SUM(DEP_CNT)/1 AS FLT1 ,SUM(ARR_CNT)/1 AS FLT2 ,SUM(DEP_CNT+ARR_CNT)/1 AS FLT3 ,SUM(DEP_SEATS)/1 AS ST1 ,SUM(DEP_SEATS)/IFEQUAL(SUM(DEP_CNT),0,1,SUM(DEP_CNT)) AS ST2 ,SUM(ARR_SEATS)/1 AS ST3 ,SUM(ARR_SEATS)/IFEQUAL(SUM(ARR_CNT),0,1,SUM(ARR_CNT)) AS ST4  FROM LOCIDSUMS WHERE YYYYMMDD IN (20250424,20250425,20250426,20250427,20250428,20250429,20250430) AND LOCID IN (' MCO') GROUP BY YYYYMMDD,LOCID,FLT_TYPE,USER_CLASS,WEIGHT_CLASS,PHYS_CLASS,ETMS_EQPT,B_JET,R_JET,BUSAVIATION,AAC,ADG,TDG ORDER BY YYYYMMDD,LOCID,FLT_TYPE,USER_CLASS,WEIGHT_CLASS,PHYS_CLASS,ETMS_EQPT,B_JET,R_JET,BUSAVIATION,AAC,ADG,TDG
cmd: tot
nopage: y
nost: n
defs:
avgdays: 1
oktosave: y
locInput_param:
locQuick_param:
locMode: on
eqptInput_param:
eqptQuick_param:
eqptMode: on
dtype: d
fm_m: 01
fy_m: 2025
tm_m: 01
ty_m: 2025
fy_y: 2025
ytype: c
ty_y: 2025
fm_r: 01
fd_r: 01
fy_r: 2025
tm_r: 01
td_r: 01
ty_r: 2025
fltype: ?
phclass: ?
wtclass: ?
usclass: ?
rjet: ?
bjet: ?
bavia: ?
reptype: tot
reportformat: asp
state:
region:
ftype:
hubsize:
isforeign:
ustower: """

#dates = ['20250424', '20250425', '20250426','20250427','20250428','20250429','20250430']
arpt_code= 'MCO' #'JFK'

generate_faa_delay_reports(
    airport_code=arpt_code,
    header_string=header_str,
    cookie_string=cookie_str,
    payload_string=payload_template_RAW,
    output_csv=f"faa_output_{arpt_code}.csv",
    wait_seconds=10
)





 Cleaned data saved to 'faa_output_MCO.csv'

 Total records saved: '705'


,#,Date,Airport,Flight Type,User Class,Weight Class,Physical Class,Aircraft Type,Business Jet,Regional Jet,...,Airplane Design Group,Taxiway Design Group,Departures,Arrivals,Total Operations,Departure Seats,Average Departure Seats,Arrival Seats,Average Arrival Seats,AirportCode
0,1,04/24/2025,MCO,Foreign to US,Air Carrier,?,Jet,E295 - unknown,No,Unknown,...,III,3,0,3,3,0,0,0,0,MCO
1,2,04/24/2025,MCO,Foreign to US,Air Carrier,Heavy Eqpt,Jet,A333 - Airbus A330-300,No,No,...,V,5,0,3,3,0,0,1005,335,MCO
2,3,04/24/2025,MCO,Foreign to US,Air Carrier,Heavy Eqpt,Jet,A339 - A330-900neo,No,Unknown,...,V,5,0,1,1,0,0,287,287,MCO
3,4,04/24/2025,MCO,Foreign to US,Air Carrier,Heavy Eqpt,Jet,A35K - Airbus A350-1000,No,Unknown,...,V,6,0,3,3,0,0,1107,369,MCO
5,5,04/24/2025,MCO,Foreign to US,Air Carrier,Heavy Eqpt,Jet,B763 - Boeing 767-300,No,No,...,IV,5,0,1,1,0,0,230,230,MCO


## TO DO:
Modify the above function so that you can specify your own date and airport_code. For example:
```
dates = ['20250424', '20250425', '20250426','20250430']
arpt_code= 'JFK'
```
Check the payload_template_RAW
